# 5. Conclusiones Globales, Supuestos de los Modelos y Auditoría de Errores

**Materia:** Aprendizaje Automático (Machine Learning)  
**Trabajo Práctico 2:** Clasificación Supervisada  
**Dataset:** Bank Marketing (UCI / Kaggle)  
**Objetivo de este cuaderno:** Consolidar las respuestas integrales y académicas solicitadas en las pautas del trabajo práctico. Este cuaderno no entrena nuevos modelos, sino que ingesta los artefactos guardados a lo largo de los Pasos 1 al 4 para responder de forma crítica y rigurosa:
1. **¿Qué modelos funcionaron mejor y por qué?** Análisis fundamentado en el compromiso entre sesgo y varianza (*Bias-Variance Trade-off*).
2. **¿Los datos satisfacen los supuestos de cada modelo?** Evaluación teórica y empírica de Gaussian Naive Bayes, Support Vector Machines, K-Nearest Neighbors y Random Forest.
3. **¿Qué clasificación de errores es más relevante para el problema de negocio?** Análisis de costos asimétricos y auditoría detallada del perfil de los **Falsos Negativos**.
4. **¿Cuáles son las limitaciones prácticas del sistema y qué mejoras se proponen?** Hoja de ruta para futuras iteraciones en producción.

In [ ]:
# =============================================================================
# 1. Configuración del Entorno y Carga de Resultados Consolidados
# =============================================================================
import os
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import roc_curve, roc_auc_score, recall_score, precision_score, f1_score, confusion_matrix

# Carpetas de almacenamiento
os.makedirs("figures", exist_ok=True)
os.makedirs("datasets", exist_ok=True)

# Configuración visual para figuras de presentación (300 DPI)
%matplotlib inline
plt.rcParams["figure.dpi"] = 100
plt.rcParams["font.size"] = 9
plt.rcParams["axes.titlesize"] = 10.5
plt.rcParams["axes.labelsize"] = 9
sns.set_theme(style="whitegrid")

# Carga de artefactos de los Pasos 2, 3 y 4
p2_path = os.path.join("datasets", "paso2_cv_results.joblib")
p3_path = os.path.join("datasets", "paso3_hiperparametros_results.joblib")
p4_path = os.path.join("datasets", "modelo_final.joblib")

res_folds_p2, oof_p2 = joblib.load(p2_path)
datos_p3 = joblib.load(p3_path)
artefacto_final = joblib.load(p4_path)

df_train = pd.read_csv("datasets/ds_train_set2_core_decorrelated.csv")
df_test = pd.read_csv("datasets/ds_test_set2_core_decorrelated.csv")

y_train = df_train["y"]
X_test = df_test.drop(columns=["y"])
y_test = df_test["y"]

print("Todos los resultados consolidados de los Pasos 1, 2, 3 y 4 cargados exitosamente.")

## 5.1 Resumen Ejecutivo del Recorrido Metodológico

A lo largo del proyecto se implementó un flujo riguroso para construir un sistema predictivo robusto y libre de fugas de información:

| Paso | Etapa | Decisiones Clave | Resultado Principal |
| :---: | :--- | :--- | :--- |
| **1** | **EDA y Preprocesamiento** | - Eliminación de `duration` (fuga temporal irreversible).<br>- Auditoría de variable objetivo ($88.7\% / 11.3\%$).<br>- Depuración de multicolinealidad macroeconómica ($VIF < 3$). | Creación del **Set 2 (Core Decorrelated - 7 variables)**. |
| **2** | **Clasificación Inicial** | - Evaluación estratificada de 4 algoritmos sobre 3 subconjuntos.<br>- Diagnóstico de sobreajuste preliminar. | Random Forest lidera en ROC-AUC ($0.753$), pero con fuerte sobreajuste en train ($0.996$). |
| **3** | **Ajuste de Hiperparámetros** | - Curvas de validación sistemáticas.<br>- Poda de profundidad en RF (`max_depth = 10`).<br>- Regularización de KNN ($k = 101$) y SVM ($C = 0.01$). | Freno drástico del sobreajuste: la brecha de RF cae de $0.243 \to 0.033$, alcanzando $\text{AUC} = \mathbf{0.801}$. |
| **4** | **Modelo Final y Test** | - Calibración de umbral óptimo sobre OOF ($\tau^* \approx 0.087$).<br>- Evaluación única en test. | Generalización confirmada: $\text{AUC} \approx 0.803$, $\text{Recall} \approx 0.683$, $\text{Lift} = \mathbf{2.7\times}$. |

## 5.2 ¿Qué modelo funcionó mejor y por qué? (Compromiso Sesgo-Varianza)

Para comparar equitativamente los cuatro modelos sintonizados en todas las métricas operativas (incluyendo Recall y Precisión), aplicamos a cada uno **su respectivo umbral óptimo** obtenido mediante la mínima distancia a $(0, 1)$ sobre sus predicciones fuera de pliegue (*OOF*).

In [ ]:
# Comparación justa de modelos sintonizados con umbral calibrado por modelo
ORDEN_MODELOS = ["Naive Bayes", "SVM", "KNN", "Random Forest"]
oof_aj = datos_p3["oof_ajustados"]

tabla_comparativa = []

for m in ORDEN_MODELOS:
    s = oof_aj[m]["score"]
    fpr, tpr, umbrales = roc_curve(y_train, s)
    dist = np.sqrt(fpr**2 + (1 - tpr)**2)
    i_opt = np.argmin(dist[1:]) + 1
    u_opt = float(umbrales[i_opt])
    pred = (s >= u_opt).astype(int)
    
    auc = roc_auc_score(y_train, s)
    rec = recall_score(y_train, pred, zero_division=0)
    prec = precision_score(y_train, pred, zero_division=0)
    pct_llamadas = pred.mean()
    
    tabla_comparativa.append({
        "Modelo": m,
        "ROC-AUC": auc,
        "Umbral Óptimo": u_opt,
        "Recall": rec,
        "Precisión": prec,
        "% Contactados": pct_llamadas,
        "Lift vs Azar": (rec / pct_llamadas) if pct_llamadas > 0 else 0
    })

df_resumen_modelos = pd.DataFrame(tabla_comparativa).set_index("Modelo")

# Visualización Comparativa de Modelos (Figura 20)
fig, axes = plt.subplots(1, 2, figsize=(13, 4.4))

# Subplot 1: Capacidad Predictiva Global (ROC-AUC y Recall)
df_resumen_modelos[["ROC-AUC", "Recall"]].plot(kind="bar", ax=axes[0], color=["#2b83ba", "#d7191c"], edgecolor="black", lw=0.6)
axes[0].set_title("Capacidad Predictiva Global (ROC-AUC y Recall)", fontsize=11, fontweight="bold")
axes[0].set_ylabel("Valor de la Métrica")
axes[0].set_ylim(0.5, 0.95)
axes[0].set_xticklabels(ORDEN_MODELOS, rotation=0, fontweight="bold")
axes[0].legend(loc="upper left")

# Subplot 2: Eficiencia Comercial (% Base a Llamar y Precisión)
df_resumen_modelos[["% Contactados", "Precisión"]].plot(kind="bar", ax=axes[1], color=["#7f7f7f", "#1b9e77"], edgecolor="black", lw=0.6)
axes[1].set_title("Eficiencia de Campaña (% Contactados y Precisión)", fontsize=11, fontweight="bold")
axes[1].set_ylabel("Proporción")
axes[1].set_ylim(0, 0.6)
axes[1].set_xticklabels(ORDEN_MODELOS, rotation=0, fontweight="bold")
axes[1].legend(loc="upper left")

plt.tight_layout()
fig.savefig("figures/20_ranking_modelos_conclusiones.png", dpi=300, bbox_inches="tight")
plt.show()

print("Figura guardada en: 'figures/20_ranking_modelos_conclusiones.png'")
df_resumen_modelos.round(3)

### Explicación Teórica del Desempeño (Compromiso Sesgo-Varianza):
1. **Random Forest (1° Puesto · $\text{ROC-AUC} = \mathbf{0.801}$):**
   - *Fundamento:* Combina el **bajo sesgo** de los árboles de decisión (capaces de trazar fronteras de decisión ortogonales no lineales) con la **reducción masiva de varianza** que otorga el ensamble *bagging* y la selección aleatoria de predictores (*Random Subspaces*).
   - Al podar la profundidad máxima a `max_depth = 10`, se eliminó la memorización de ruido, alcanzando el equilibrio óptimo con una brecha de generalización mínima ($0.033$).
2. **KNN (2° Puesto · $\text{ROC-AUC} = \mathbf{0.787}$):**
   - *Fundamento:* Al ser un método no paramétrico local, KNN carece de sesgo sobre la forma de la frontera. Sin embargo, con $k=5$ sufría de altísima varianza estocástica. Al incrementar a $k=101$ con ponderación uniforme, se estabilizó la estimación de la densidad local de la clase minoritaria, logrando un notable desempeño competitivo.
3. **Naive Bayes (3° Puesto · $\text{ROC-AUC} = 0.753$):**
   - *Fundamento:* Modelo de **alto sesgo** e inflexibilidad estructural. Su frontera de decisión es cuadrática y rígida. Aunque es inmune a la varianza y rápida de entrenar, no puede superar el límite impuesto por sus fuertes supuestos probabilísticos.
4. **SVM (4° Puesto · $\text{ROC-AUC} = 0.708$):**
   - *Fundamento:* La maximización del margen de separación resulta intrínsecamente problemática en presencia de **fuerte solapamiento de clases y desbalance severo**. Con un margen suave ($C = 0.01$), el modelo mejora pero carece de granularidad para generar un ordenamiento probabilístico refinado en problemas con fronteras difusas.

## 5.3 ¿Los datos cumplen los supuestos de cada modelo?

Examinamos cómo las propiedades empíricas de los datos de Bank Marketing desafían los supuestos fundacionales de cada familia algorítmica:

| Propiedad de los Datos (Evidencia Empírica) | Gaussian Naive Bayes | Support Vector Machines (SVM) | K-Nearest Neighbors (KNN) | Random Forest |
| :--- | :--- | :--- | :--- | :--- |
| **Dependencia y Correlación entre Variables**<br>*(EDA: Correlación macro $> 0.90$)* | ❌ **Viola gravemente** la independencia condicional $P(x \mid y) = \prod P(x_i \mid y)$. Tiende a sobreestimar certezas. | ✔ Poco afectado; el hiperplano pondera las combinaciones lineales. | ⚠️ Distorsiona las distancias ponderando dos veces la misma dimensión latente. | ✔ **Completamente robusto**; las particiones individuales descartan redundancias. |
| **Distribuciones No Gaussianas y Asimétricas**<br>*(EDA: Asimetrías en `campaign`, `previous` y binarizaciones)* | ❌ **Viola el supuesto** de normalidad en clases continuas. | ✔ **No asume distribución paramétrica**. | ✔ **No asume distribución paramétrica**. | ✔ **No asume distribución paramétrica**. |
| **Escalas y Rangos Dispares**<br>*(EDA: `euribor3m` de $0.6$ a $5.0$ vs `previous` de $0$ a $7$)* | ✔ Invariante; calcula medias y varianzas independientes por variable. | ⚠️ **Sensible a la escala**; exige `StandardScaler` pliegue a pliegue. | ⚠️ **Altamente sensible**; las variables con mayor rango dominarían las distancias euclidianas. | ✔ **Estrictamente invariante** a transformaciones monótonas de escala. |
| **Fronteras Altamente No Lineales**<br>*(EDA: Relaciones en "U" en la edad, efectos estacionales)* | ⚠️ Capacidad limitada a fronteras cuadráticas. | ⚠️ Requiere necesariamente kernel no lineal (**RBF**); el kernel lineal subajustó severamente ($0.609$). | ✔ Frontera local no paramétrica muy flexible. | ✔ **Excelente capacidad** para modelar interacciones no lineales de orden superior. |
| **Desbalance Severo de Clases ($11.3\%$ positivas)**<br>*(EDA: Fuerte solapamiento)* | ✔ Incorpora el prior $P(y)$, pero genera probabilidades extremas. | ⚠️ El margen queda dominado por la clase mayoritaria. | ⚠️ $k$ grande diluye la densidad minoritaria; exige umbral bajo. | ⚠️ Pocas hojas alcanzan mayoría; exige calibración de umbral $\tau^*$. |

> [!NOTE]
> **Conclusión sobre los supuestos:** Random Forest es el clasificador que mejor se adapta al problema debido a que es el único que no asume independencia, normalidad, linealidad ni homogeneidad de escalas.

## 5.4 Auditoría de Errores: ¿Qué clasificación de errores es más relevante?

### Asimetría Crítica de Errores:
- Como se fundamentó en el diseño comercial, el **Falso Negativo (FN)** es el error más costoso: representa clientes con voluntad de contratar a los que el banco no contacta por decisión del modelo.
- En el conjunto de test independiente, el modelo final captura al **$68.3\%$** de los interesados (635 depósitos), pero **pierde a 295 interesados ($31.7\%$)**.

### Auditoría Empírica: ¿Quiénes son los clientes no detectados (Falsos Negativos)?
Para descubrir si los errores ocurren al azar o si el modelo presenta sesgos estructurales sistemáticos, auditamos el perfil de los clientes clasificados como Falsos Negativos en test.

In [ ]:
# Segmentación y auditoría de los clientes que verdaderamente querían suscribir (y = 1 en Test)
modelo_final = artefacto_final["modelo"]
umbral_final = artefacto_final["umbral"]

score_test = modelo_final.predict_proba(X_test)[:, 1]
pred_test = (score_test >= umbral_final).astype(int)

df_interesados = df_test[df_test["y"] == 1].copy()
df_interesados["detectado"] = (score_test[df_test["y"] == 1] >= umbral_final).astype(int)

# Segmentación por Nivel de Tasas de Interés (euribor3m)
df_interesados["clima_tasas"] = pd.cut(
    df_interesados["euribor3m"],
    bins=[-np.inf, 2.0, 4.0, np.inf],
    labels=["Tasas Bajas (< 2.0)", "Tasas Medias (2.0 - 4.0)", "Tasas Altas (> 4.0)"]
)

recall_por_tasas = df_interesados.groupby("clima_tasas", observed=False)["detectado"].agg(
    total="count",
    detectados="sum",
    recall="mean"
)

# Segmentación por Historial de Campaña Previa (poutcome_success)
recall_por_historial = df_interesados.groupby("poutcome_success", observed=False)["detectado"].agg(
    total="count",
    detectados="sum",
    recall="mean"
)
recall_por_historial.index = ["Sin éxito previo (0)", "Con éxito previo (1)"]

# Visualización de la Auditoría de Falsos Negativos (Figura 21)
fig, axes = plt.subplots(1, 2, figsize=(13, 4.4))

# Subplot 1: Tasa de Detección según Clima de Tasas de Interés
recall_por_tasas["recall"].plot(kind="bar", ax=axes[0], color=["#1b9e77", "#fdae61", "#d7191c"], edgecolor="black", lw=0.6)
for idx, (val, tot) in enumerate(zip(recall_por_tasas["recall"], recall_por_tasas["total"])):
    axes[0].text(idx, val + 0.02, f"{val:.1%}\n(n={tot})", ha="center", fontsize=8.5, fontweight="bold")
axes[0].set_title("Tasa de Detección de Interesados según Tasas de Interés", fontsize=11, fontweight="bold")
axes[0].set_ylabel("Recall en el Segmento")
axes[0].set_ylim(0, 1.15)
axes[0].set_xticklabels(recall_por_tasas.index, rotation=0, fontweight="bold")

# Subplot 2: Tasa de Detección según Historial de Campañas
recall_por_historial["recall"].plot(kind="bar", ax=axes[1], color=["#2b83ba", "#1b9e77"], edgecolor="black", lw=0.6)
for idx, (val, tot) in enumerate(zip(recall_por_historial["recall"], recall_por_historial["total"])):
    axes[1].text(idx, val + 0.02, f"{val:.1%}\n(n={tot})", ha="center", fontsize=8.5, fontweight="bold")
axes[1].set_title("Tasa de Detección de Interesados según Historial Previo", fontsize=11, fontweight="bold")
axes[1].set_ylabel("Recall en el Segmento")
axes[1].set_ylim(0, 1.15)
axes[1].set_xticklabels(recall_por_historial.index, rotation=0, fontweight="bold")

plt.tight_layout()
fig.savefig("figures/21_auditoria_errores_falsos_negativos.png", dpi=300, bbox_inches="tight")
plt.show()

print("Figura guardada en: 'figures/21_auditoria_errores_falsos_negativos.png'")
print("\nAuditoría de Detección por Nivel de Tasas:")
recall_por_tasas.round(3)

### Hallazgos Clave de la Auditoría de Falsos Negativos:
1. **El contexto macroeconómico oculta a los interesados individuales:**
   - En periodos de **tasas bajas o intermedias**, el modelo detecta a más del **$80\%$** de los clientes interesados.
   - En cambio, en periodos de **tasas altas (`euribor3m > 4.0`)**, donde la tasa base promedio de suscripción fue históricamente deprimida durante la crisis, el modelo tiende a clasificar de forma conservadora a toda la población del periodo como negativa, provocando que la mayoría de los Falsos Negativos provengan de esta coyuntura.
2. **Impacto del Historial Comercial:**
   - Para clientes con éxito en campañas anteriores (`poutcome_success = 1`), la tasa de detección es prácticamente perfecta ($> 90\%$).
   - La dificultad se concentra en clientes sin historial comercial previo, donde la ausencia de antecedentes fuerza al modelo a apoyarse casi exclusivamente en las variables macroeconómicas.

## 5.5 Limitaciones del Sistema Actual

1. **Dependencia Temporal del Ciclo Económico:**  
   El dataset proviene del periodo 2008–2010 (crisis financiera en Portugal/Europa). Debido a que el split train/test fue aleatorio estratificado, ambos conjuntos comparten los mismos periodos temporales. Si el modelo se aplica a una campaña futura en un ciclo económico muy distinto, su rendimiento podría degradarse.
2. **Exclusión Metodológica de `duration`:**  
   Aunque imprescindible para evitar *data leakage*, prescindir de la duración impone un límite predictivo natural al problema ($\text{ROC-AUC} \approx 0.80$), dado que las variables demográficas y de contacto no alcanzan una separación total.
3. **Criterio de Umbral Geométrico vs. Matriz Monetaria Real:**  
   El umbral se seleccionó mediante la distancia euclidiana en el espacio ROC, en lugar de optimizar una función de costos en euros reales ($\text{Costo de llamada}$ vs. $\text{Beneficio por colocación de depósito}$), la cual no está especificada en el dataset de origen.
4. **Falta de Variables Transaccionales del Cliente:**  
   El banco no dispone en este dataset de saldos en cuenta corriente, historial de movimientos o tenencia de otros productos (tarjetas, hipotecas), variables que enriquecerían el perfil del cliente por sobre el contexto macroeconómico.

## 5.6 Hoja de Ruta y Mejoras Propuestas

| Mejora Propuesta | Justificación Teórica y de Negocio | Viabilidad en Producción |
| :--- | :--- | :---: |
| **Optimización de Umbral por Utilidad Monetaria** | Sustituir el criterio geométrico por la maximización del beneficio financiero esperado: $\max_\tau [\text{TP}(\tau) \cdot V_{\text{depósito}} - (\text{TP}(\tau) + \text{FP}(\tau)) \cdot C_{\text{llamada}}]$. | Inmediata (con datos de costos del banco). |
| **Validación Cruzada Temporal (*Time-Series Split*)** | Entrenar con los primeros $N$ meses y evaluar en los meses subsiguientes para auditar la resistencia al cambio de ciclo macroeconómico. | Muy Alta. |
| **Ensamble Híbrido (*Blending* RF + KNN)** | Combinar las probabilidades de Random Forest (reglas globales ortogonales) con KNN ($k=101$, densidades locales continuas) para suavizar las fronteras en casos dudosos. | Alta. |
| **Modelos Segmentados por Régimen de Tasas** | Entrenar clasificadores especializados para escenarios de tasas bajas y tasas altas, recuperando a los clientes interesados en periodos restrictivos. | Media. |
| **Enriquecimiento del Perfil de Cliente** | Incorporar saldo promedio mensual, nivel salarial y antigüedad bancaria para desanclar al modelo de su dependencia del mes y del Euribor. | Requiere ingesta de nuevas fuentes del Data Warehouse. |

## 5.7 Mensaje Final de Cierre

- **Solución Definitiva:** El clasificador **Random Forest (`max_depth = 10`, `n_estimators = 50`)** operando sobre el **Set 2 (Core Decorrelated)** con un umbral de decisión calibrado en **$\tau^* \approx 0.087$** constituye la solución analítica óptima para el problema de Bank Marketing.
- **Rendimiento Comprobado:** Obtiene un **$\text{ROC-AUC} \approx 0.803$** en test y rescata al **$68.3\%$** de los clientes dispuestos a contratar un depósito a plazo fijo, contactando a solo el **$25.1\%$** de la base.
- **Impacto de Negocio:** Multiplica la tasa de éxito por llamada en **$2.7\times$** frente a una campaña no dirigida, ahorrando el $75\%$ de los costos operativos telefónicos del banco y garantizando un proceso analítico transparente, robusto y metodológicamente validado.